
This notebook is meant to run periodically, to assess the "topics" that appear in the error messages.

We'll be using SparkML, so you need to be running this notebook on a dedicated cluster (not serverless, not shared). For simplicity, we'll use K-means to do the clustering.

In [0]:

dbutils.widgets.text(
  "target_catalog", "field_demos", "Target Catalog"
  )
dbutils.widgets.text(
  "target_schema", "tauherng", "Target Schema"
  )
dbutils.widgets.text(
  "target_table", "job_errors", "Target Table"
  )
dbutils.widgets.text(
  "embeddings_table", "job_errors_embeddings", "Embeddings Table"
  )


Databricks AI Functions provides an easy way to calculate embeddings directly using SQL. Pretty convenient, in my opinion!

No more need to mess around with provisioning an embedding model, setting up a serving endpoint, or any of that. Just point it to the embedding model, pass in the columns, and away we go.

In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(CONCAT(:target_catalog, ".", :target_schema, ".", :embeddings_table)) AS SELECT
  *,
  ai_query("databricks-bge-large-en", -- embedding model
  error || regexp_replace(coalesce(error_trace, ""), '[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}', '<uuid>') -- columns I want to calculate embeddings for
  ) as embedding
FROM
  IDENTIFIER(CONCAT(:target_catalog, ".", :target_schema, ".", :target_table));


Now we can run it through a simple k-means classifier!

In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import array_to_vector

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
embeddings_table = dbutils.widgets.get("embeddings_table")

embeddings_df = spark.table(f"{target_catalog}.{target_schema}.{embeddings_table}")
embeddings_df = embeddings_df.withColumn("features", array_to_vector("embedding")) # already an array, just need to convert it to vector for input into the KMeans function

kmeans = KMeans(featuresCol="features", k=15, seed=1) # I assume there will be ~20 common error types
model = kmeans.fit(embeddings_df)
clustered_df = model.transform(embeddings_df).withColumnRenamed("prediction", "cluster")

In [0]:
spark.sql(
    """
          SELECT cluster, count(run_id) as cluster_size from {clustered_df}
          GROUP BY cluster
          ORDER BY `cluster` desc""",
    clustered_df=clustered_df,
).show(30)

In [0]:
# Downsample for each cluster (n <= 50)
from pyspark.sql.functions import col
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.functions import to_json

window = Window.partitionBy("cluster").orderBy(F.rand())
downsampled_df = clustered_df.withColumn("row_num", F.row_number().over(window)).filter(col("row_num") <= 50).drop("row_num")

# Then collect all the error logs for each cluster
logs_per_cluster_df = downsampled_df.groupBy("cluster").agg(F.collect_list("error").alias("sample_logs"))

logs_per_cluster_df = logs_per_cluster_df.withColumn("logs_json", to_json("sample_logs"))

# Then run a summary of each cluster
# Here we simplistically use AI_QUERY with a custom prompt, to use a Databricks-hosted LLM to summarize the sampled errors for each cluster
logs_per_cluster_df.withColumn("cluster_description", F.expr(
  """
  ai_query('databricks-gpt-oss-120b', -- can swap another model if preferred
  "I'm clustering some error logs. You will get a sample of the messages from one of the clusters. Give me a cluster topic based on the messages that is at most 10 words, preferably less than 5:" || logs_json)
  """
)).drop("logs_json").write.mode('overwrite').saveAsTable(f"{target_catalog}.{target_schema}.job_error_cluster_descriptions")

In [0]:
%sql
SELECT * FROM IDENTIFIER(CONCAT(:target_catalog,".",:target_schema,".job_error_cluster_descriptions"));